In [1]:
import pandas as pd
import numpy as np
import seaborn as sns


from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings


In [2]:
data = pd.read_csv('data/stud_eda.csv')

In [3]:
x= data.drop(columns='math_score' , axis=1)
y= data['math_score']

In [84]:
df = data
print("Categories in 'gender' variable:     ",end=" " )
print(df['gender'].unique())

print("Categories in 'race_ethnicity' variable:  ",end=" ")
print(df['race_ethnicity'].unique())

print("Categories in'parental level of education' variable:",end=" " )
print(df['parental_level_of_education'].unique())

print("Categories in 'lunch' variable:     ",end=" " )
print(df['lunch'].unique())

print("Categories in 'test preparation course' variable:     ",end=" " )
print(df['test_preparation_course'].unique())

Categories in 'gender' variable:      ['female' 'male']
Categories in 'race_ethnicity' variable:   ['group B' 'group C' 'group A' 'group D' 'group E']
Categories in'parental level of education' variable: ["bachelor's degree" 'some college' "master's degree" "associate's degree"
 'high school' 'some high school']
Categories in 'lunch' variable:      ['standard' 'free/reduced']
Categories in 'test preparation course' variable:      ['none' 'completed']


In [4]:
int_feature = [i for i in x.columns if x[i].dtype != 'O']
cat_feature = [i for i in x.columns if x[i].dtype == 'O']

In [86]:
from sklearn.preprocessing import LabelEncoder ,OrdinalEncoder ,OneHotEncoder, StandardScaler
onehot = OneHotEncoder(drop='first')
label = LabelEncoder()
oridinal = OrdinalEncoder()
scaler = StandardScaler()


In [87]:
data.nunique()

Unnamed: 0                     1000
gender                            2
race_ethnicity                    5
parental_level_of_education       6
lunch                             2
test_preparation_course           2
math_score                       81
reading_score                    72
writing_score                    77
total_marks                     194
average_marks                   194
dtype: int64

In [88]:


one_hot_arr = ['parental_level_of_education']
label_arr = ['gender' , 'lunch' ,'test_preparation_course']
oridinal_arr = ['race_ethnicity']


In [89]:
# Apply Label Encoding to label_arr columns
for col in label_arr:
    x[col] = label.fit_transform(x[col].astype(str))

In [90]:
# Apply Ordinal Encoding to oridinal_arr columns
for col in oridinal_arr:
    x[col] = oridinal.fit_transform(x[[col]])

In [91]:
df.head()

,Unnamed: 0,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,total_marks,average_marks
0,0,female,group B,bachelor's degree,standard,none,72,72,74,218,72.666667
1,1,female,group C,some college,standard,completed,69,90,88,247,82.333333
2,2,female,group B,master's degree,standard,none,90,95,93,278,92.666667
3,3,male,group A,associate's degree,free/reduced,none,47,57,44,148,49.333333
4,4,male,group C,some college,standard,none,76,78,75,229,76.333333


In [92]:


from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", onehot, one_hot_arr),
        ("StandardScaler", scaler, int_feature)
    ], remainder='passthrough'
)


In [93]:
# Check the encoded data
df.drop(columns='Unnamed: 0' , axis=1 , inplace=True)

In [94]:
x = preprocessor.fit_transform(x)

In [95]:
x

array([[1., 0., 0., ..., 1., 1., 1.],
       [0., 0., 0., ..., 2., 1., 0.],
       [0., 0., 1., ..., 1., 1., 1.],
       ...,
       [0., 1., 0., ..., 2., 0., 0.],
       [0., 0., 0., ..., 3., 1., 0.],
       [0., 0., 0., ..., 3., 0., 1.]], shape=(1000, 14))

In [97]:
# train test split
from sklearn.model_selection import train_test_split
x_train , x_test , y_train , y_test = train_test_split(x , y , train_size=0.8 , random_state=45)

In [99]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [101]:
model_name= {

'LinearRegression':LinearRegression(),
'Ridge':Ridge(),
'Lasso':Lasso(),
'KNeighborsRegressor' : KNeighborsRegressor(),
'DecisionTreeRegressor': DecisionTreeRegressor(),
'RandomForestRegressor':RandomForestRegressor(),
'AdaBoostRegressor' : AdaBoostRegressor(),
'SVR': SVR(),
'XGBRegressor':XGBRegressor()
}

model_list = []
r2_list =[]

for i in range(len(list(model_name))):
    model = list(model_name.values())[i]
    model.fit(x_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(model_name.keys())[i])
    model_list.append(list(model_name.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

LinearRegression
Model performance for Training set
- Root Mean Squared Error: 0.0000
- Mean Absolute Error: 0.0000
- R2 Score: 1.0000
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.0000
- Mean Absolute Error: 0.0000
- R2 Score: 1.0000


Ridge
Model performance for Training set
- Root Mean Squared Error: 0.3297
- Mean Absolute Error: 0.2625
- R2 Score: 0.9995
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.3315
- Mean Absolute Error: 0.2662
- R2 Score: 0.9995


Lasso
Model performance for Training set
- Root Mean Squared Error: 4.6731
- Mean Absolute Error: 3.6724
- R2 Score: 0.9065
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 4.6411
- Mean Absolute Error: 3.7729
- R2 Score: 0.8979


KNeighborsRegressor
Model performance for Training set
- Root Mean Squared Error: 4.6507
- Mean Absolute Error: 3.6455
- R2 Score: 0.9074
--------------------------

In [102]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

,Model Name,R2_Score
0,LinearRegression,1.000000
1,Ridge,0.999479
8,XGBRegressor,0.971560
5,RandomForestRegressor,0.955637
6,AdaBoostRegressor,0.917713
4,DecisionTreeRegressor,0.916753
2,Lasso,0.897885
7,SVR,0.873505
3,KNeighborsRegressor,0.857632
